# Prophet：Facebook 的时间序列利器
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：10_时间序列 | 源文件：Prophet.py | 核心功能：趋势+季节+节假日分解、自动预测

## 概述

Prophet 是 Facebook 开发的时间序列预测工具，设计目标是让非专家也能做出高质量预测。它将时间序列分解为趋势 + 季节性 + 节假日效应。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 数学原理

### 1. Prophet 的分解模型

Prophet 将时间序列分解为三个成分：

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

- $g(t)$：趋势项（增长曲线）
- $s(t)$：周期性季节项
- $h(t)$：节假日/事件效应
- $\epsilon_t$：误差项（假设服从正态分布）

### 2. 趋势模型

**线性趋势**：

$$g(t) = (k + \mathbf{a}(t)^\top \delta) t + (m + \mathbf{a}(t)^\top \gamma)$$

其中 $k$ 是基础增长率，$\delta$ 是速率调整量，$m$ 是偏移量。

**逻辑增长趋势**（饱和增长）：

$$g(t) = \frac{C(t)}{1 + \exp(-(k + \mathbf{a}(t)^\top \delta)(t - (m + \mathbf{a}(t)^\top \gamma)))}$$

其中 $C(t)$ 是承载能力（可随时间变化）。

**变点检测**：Prophet 自动检测趋势变化点 $\{s_1, \ldots, s_S\}$，在每个变点处允许增长率变化。

### 3. 季节性模型（傅里叶级数）

用傅里叶级数拟合周期为 $P$ 的季节性：

$$s(t) = \sum_{n=1}^{N}\left(a_n \cos\left(\frac{2\pi n t}{P}\right) + b_n \sin\left(\frac{2\pi n t}{P}\right)\right)$$

- 年季节性：$P=365.25$，$N=10$（默认）
- 周季节性：$P=7$，$N=3$（默认）

$N$ 越大，季节性曲线越灵活（可能过拟合）。

### 4. 节假日/事件效应

$$h(t) = \sum_{i=1}^{L} \kappa_i \cdot \mathbb{1}[t \in D_i]$$

其中 $D_i$ 是第 $i$ 个节假日的影响窗口（前后若干天），$\kappa_i$ 是待学习的效应大小。

### 5. 拟合方法

Prophet 使用 **Stan** 后端进行最大后验估计（MAP）：

$$\theta^* = \arg\max_\theta \left[\log P(y|\theta) + \log P(\theta)\right]$$

- 趋势参数的先验：变点数量、增长率的正则化
- 季节性参数的先验：傅里叶系数的正则化

### 6. 不确定性估计

Prophet 提供预测区间，通过：
- 趋势的不确定性：模拟未来变点的可能位置和大小
- 季节性的不确定性：通过贝叶斯后验采样

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `Prophet()` | 构建分解模型 $y = g + s + h + \epsilon$ |
| `df["ds"]` | 时间戳 $t$ |
| `df["y"]` | 观测值 $y(t)$ |
| `model.fit(train)` | MAP 估计所有参数 |
| `model.predict(future)` | 输出 $g(t), s(t), h(t)$ 及预测值 |
| `forecast["yhat"]` | $\hat{y}(t) = g(t) + s(t) + h(t)$ |
| `forecast["yhat_lower/upper"]` | 预测区间（不确定性） |
| `add_seasonality(name, period, fourier_order)` | 添加傅里叶季节性，$P$ 和 $N$ |

### 1. 生成合成时间序列

运行 1. 生成合成时间序列 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 365 * 3  # 三年日数据

t = np.arange(n)
# 非线性趋势（饱和增长）
趋势 = 10 * np.log1p(t / 50)
# 年度季节性
年季节 = 2.5 * np.sin(2 * np.pi * t / 365) + 1.5 * np.cos(4 * np.pi * t / 365)
# 周季节性
周季节 = np.where(pd.Series(t).apply(lambda x: pd.Timestamp("2021-01-01") + pd.Timedelta(days=x)).dt.dayofweek >= 5, -1.5, 0)
噪声 = np.random.normal(0, 0.4, n)
数据 = 趋势 + 年季节 + 周季节 + 噪声

# 构造 Prophet 要求的 DataFrame 格式（ds + y）
dates = pd.date_range("2021-01-01", periods=n, freq="D")
df = pd.DataFrame({"ds": dates, "y": 数据})

print("=" * 55)
print("Facebook Prophet 时间序列预测")
print("=" * 55)
print(f"数据范围: {dates[0].date()} ~ {dates[-1].date()}, 共 {n} 天")
print(f"均值: {df['y'].mean():.4f}, 标准差: {df['y'].std():.4f}")

### 2. 训练/测试划分

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
train_size = int(n * 0.8)
train = df.iloc[:train_size].copy()
test = df.iloc[train_size:].copy()
print(f"训练集: {len(train)} 天, 测试集: {len(test)} 天")

### 3. 自定义节假日

运行 3. 自定义节假日 的代码，观察算法在该环节的行为。

In [ ]:
# 构造几个模拟"假期"
节假日 = pd.DataFrame({
    "holiday": "春节",
    "ds": pd.to_datetime(["2021-02-12", "2022-02-01", "2023-01-22"]),
    "lower_window": -3,
    "upper_window": 3,
# --- 继续 ---
})
国庆 = pd.DataFrame({
    "holiday": "国庆",
    "ds": pd.to_datetime(["2021-10-01", "2022-10-01", "2023-10-01"]),
    "lower_window": -1,
# --- 继续 ---
    "upper_window": 7,
})
所有节假日 = pd.concat([节假日, 国庆], ignore_index=True)

### 4. 模型拟合

运行 4. 模型拟合 的代码，观察算法在该环节的行为。

In [ ]:
print("\n--- 模型拟合 ---")
模型 = Prophet(
    yearly_seasonality=True,   # 年季节性
    weekly_seasonality=True,   # 周季节性
    daily_seasonality=False,   # 日粒度数据不需要日季节性
# --- 赋值/配置 ---
    holidays=所有节假日,         # 自定义节假日
    changepoint_prior_scale=0.05,  # 趋势灵活性（越大越灵活）
    seasonality_prior_scale=10.0,
)
模型.fit(train)
# --- 输出结果 ---
print(f"  变点数: {len(模型.changepoints)}")
print(f"  趋势变点（前 5 个）: {[d.date() for d in 模型.changepoints[:5]]}")

### 5. 预测

使用训练好的模型进行预测，观察输出结果。

In [ ]:
future = 模型.make_future_dataframe(periods=len(test))
forecast = 模型.predict(future)
预测 = forecast.iloc[train_size:].copy()

### 6. 评估

在测试集上评估模型性能，关注关键指标。

In [ ]:
rmse = np.sqrt(mean_squared_error(test["y"], 预测["yhat"]))
mae = mean_absolute_error(test["y"], 预测["yhat"])
mape = np.mean(np.abs((test["y"].values - 预测["yhat"].values) / test["y"].values)) * 100

print(f"\n--- 预测评估 ---")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  MAPE: {mape:.2f}%")

print(f"\n--- 预测 vs 真实（前 10 个） ---")
print(f"{'日期':>12}  {'真实':>8}  {'预测':>8}  {'下界':>8}  {'上界':>8}")
for i in range(min(10, len(test))):
    print(f"  {str(test['ds'].iloc[i].date()):>10}  "
          f"{test['y'].iloc[i]:>8.3f}  "
# --- 继续 ---
          f"{预测['yhat'].iloc[i]:>8.3f}  "
          f"{预测['yhat_lower'].iloc[i]:>8.3f}  "
          f"{预测['yhat_upper'].iloc[i]:>8.3f}")

### 7. 趋势与季节性分解

运行 7. 趋势与季节性分解 的代码，观察算法在该环节的行为。

In [ ]:
print("\n--- 模型分解 ---")
趋势成分 = 预测["trend"]
年季节成分 = 预测["yearly"]
周季节成分 = 预测["weekly"]
节假日成分 = 预测["holidays"]

print(f"  趋势范围:     [{趋势成分.min():.3f}, {趋势成分.max():.3f}]")
print(f"  年季节范围:   [{年季节成分.min():.3f}, {年季节成分.max():.3f}]")
print(f"  周季节范围:   [{周季节成分.min():.3f}, {周季节成分.max():.3f}]")
print(f"  节假日效应:   [{节假日成分.min():.3f}, {节假日成分.max():.3f}]")

### 8. 未来预测（全量模型）

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n--- 全量模型拟合 & 未来 60 天预测 ---")
全模型 = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    holidays=所有节假日,
# --- 赋值/配置 ---
    changepoint_prior_scale=0.05,
)
全模型.fit(df)
未来 = 全模型.make_future_dataframe(periods=60)
未来预测 = 全模型.predict(未来)
# --- 继续 ---
未来部分 = 未来预测.tail(60)

print(f"{'日期':>12}  {'预测值':>8}  {'下界':>8}  {'上界':>8}")
for i in range(60):
    row = 未来部分.iloc[i]
    print(f"  {str(row['ds'].date()):>10}  {row['yhat']:>8.3f}  {row['yhat_lower']:>8.3f}  {row['yhat_upper']:>8.3f}")

## 关键代码解释

```python
from prophet import Prophet
m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m.fit(df)  # df 有 ds (日期) 和 y (值) 列
future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)
```

## 注意事项

1. 需要安装 `prophet` 库
2. 对突变事件敏感
3. 不适合高频率数据（秒级/分钟级）

## 总结与延伸

以上代码展示了 **Prophet** 的完整流程。

**解读要点**：
- 观察预测曲线与实际值的 **趋势一致性**
- 关注残差是否呈现随机分布（无明显模式）
- 对比不同模型的 **MAPE / RMSE** 指标

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **NeuralProphet**：Prophet 的深度学习版本
- **自动时间序列**：AutoTS、AutoGluon-TimeSeries
- **时序大模型**：TimesFM、Chronos 等基础模型
